# 01 · 并发实验

目标：观察有界 worker、channel 关闭和 context timeout。对应文档：`01_language_basics/1.3-concurrency.md`。

## 从 TypeScript 迁移

Goroutine 类似轻量任务，但不是 Promise。Channel 传递值和所有权；`context.Context` 类似可向下传播的 AbortSignal，同时携带 deadline。

In [ ]:
import (
    "context"
    "fmt"
    "sync"
    "time"
)

## 实验一：有界 Worker

先预测哪些输出顺序稳定。Worker 获取任务的顺序不确定，但所有任务只处理一次，且主流程等待 worker 后才关闭 results。

In [ ]:
%%
jobs := make(chan int, 4)
results := make(chan int, 4)
var workers sync.WaitGroup

for id := 0; id < 2; id++ {
    workers.Add(1)
    go func(workerID int) {
        defer workers.Done()
        for job := range jobs {
            fmt.Printf("worker=%d job=%d\n", workerID, job)
            results <- job * job
        }
    }(id)
}

for job := 1; job <= 4; job++ { jobs <- job }
close(jobs)
workers.Wait()
close(results)
for result := range results { fmt.Println("result", result) }

发送方负责关闭 channel。这里 `jobs` 由生产者关闭；`results` 要等所有 worker 退出后再关闭，否则仍在发送的 goroutine 会 panic。

## 实验二：取消传播

任务需要 200ms，context 预算只有 50ms。预测 select 会进入哪个分支。

In [ ]:
%%
ctx, cancel := context.WithTimeout(context.Background(), 50*time.Millisecond)
defer cancel()

select {
case <-time.After(200 * time.Millisecond):
    fmt.Println("finished")
case <-ctx.Done():
    fmt.Println("canceled:", ctx.Err())
}

## 用 Race Detector 验证真实代码

并发代码“运行一次没报错”不能证明安全。下面运行 Capstone 的队列和有界并发测试，并开启数据竞争检测。

In [ ]:
!*cd "$GO_LEARNING_CAPSTONE" && go test -race ./internal/gateway/provider -run 'TestQueue|TestBounded' -v

## 动手挑战

1. 把 jobs 改成无缓冲 channel，解释吞吐和背压变化。
2. 在 worker 中加入 context，超时后停止接收新任务。
3. 故意提前关闭 results，观察 panic 并解释所有权错误。

## 复盘

- 谁应关闭 channel？
- WaitGroup 解决生命周期还是数据竞争？
- 为什么生产代码不应靠 `time.Sleep` 等待 goroutine？